<a href="https://colab.research.google.com/github/Asav23/Text-Summariser/blob/main/SemanticClusterer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install hdbscan

In [ ]:
!pip install hdbscan

import hdbscan
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize

class SemanticClusterer:
    def __init__(self, method="hdbscan", min_cluster_size=2):
        self.method = method
        self.min_cluster_size = min_cluster_size

    def dynamic_n_clusters(self, n_ideas):
        # Scale clusters based on how many ideas there are
        if n_ideas <= 6:
            return 2
        elif n_ideas <= 12:
            return 3
        elif n_ideas <= 20:
            return 4
        else:
            return 5

    def cluster(self, embeddings):
        # Normalise embeddings so distance = cosine distance
        normalized = normalize(embeddings)

        if self.method == "hdbscan":
            clusterer = hdbscan.HDBSCAN(
                min_cluster_size=self.min_cluster_size,
                metric="euclidean"  # euclidean on normalised = cosine
            )
            labels = clusterer.fit_predict(normalized)

            # If too many ideas are discarded as noise, fall back to KMeans
            noise_ratio = np.sum(labels == -1) / len(labels)
            if noise_ratio > 0.3:  # more than 30% discarded
                n = self.dynamic_n_clusters(len(embeddings))
                clusterer = KMeans(n_clusters=n, random_state=42)
                labels = clusterer.fit_predict(normalized)

        else:
            n = self.dynamic_n_clusters(len(embeddings))
            clusterer = KMeans(n_clusters=n, random_state=42)
            labels = clusterer.fit_predict(normalized)

        return labels

/usr/local/lib/python3.12/dist-packages/hdbscan/robust_single_linkage_.py:175: SyntaxWarning: invalid escape sequence '\{'
  $max \{ core_k(a), core_k(b), 1/\alpha d(a,b) \}$.
